#### near_workspaces (test)

In [ ]:
///- --test static --symbol

:< testing

In [1]:
:< rust_operators

:! near_token_workspaces =
    `(
        rust::rust_type
            `($'near_workspaces_types_NearToken')
            `($'near_workspaces::types::NearToken')
            "type near_workspaces_types_NearToken = class end"
    )

:! gas =
    `(
        rust::rust_type
            `($'near_workspaces_types_Gas')
            `($'near_workspaces::types::Gas')
            "type near_workspaces_types_Gas = class end"
    )

:! near_workspaces_error =
    `(
        rust::rust_type
            `($'near_workspaces_error_Error')
            `($'near_workspaces::error::Error')
            "type near_workspaces_error_Error = class end"
    )

:! sandbox =
    `(
        rust::rust_type
            `($'near_workspaces_network_Sandbox')
            `($'near_workspaces::network::Sandbox')
            "type near_workspaces_network_Sandbox = class end"
    )

:! worker t =
    `(
        rust::rust_type
            `($'near_workspaces_Worker<`t>')
            `($'near_workspaces::Worker<`t>')
            "type near_workspaces_Worker<'T> = class end"
    )

:! contract =
    `(
        rust::rust_type
            `($'near_workspaces_Contract')
            `($'near_workspaces::Contract')
            "type near_workspaces_Contract = class end"
    )

:! call_transaction =
    `(
        rust::rust_type
            `($'near_workspaces_operations_CallTransaction')
            `($'near_workspaces::operations::CallTransaction')
            "type near_workspaces_operations_CallTransaction = class end"
    )

:! execution_final_result =
    `(
        rust::rust_type
            `($'near_workspaces_result_ExecutionFinalResult')
            `($'near_workspaces::result::ExecutionFinalResult')
            "type near_workspaces_result_ExecutionFinalResult = class end"
    )

:! execution_result t =
    `(
        rust::rust_type
            `($'near_workspaces_result_ExecutionResult<`t>')
            `($'near_workspaces::result::ExecutionResult<`t>')
            "type near_workspaces_result_ExecutionResult<'T> = class end"
    )

:! execution_success =
    `(
        rust::rust_type
            `($'near_workspaces_result_ExecutionSuccess')
            `($'near_workspaces::result::ExecutionSuccess')
            "type near_workspaces_result_ExecutionSuccess = class end"
    )

:! execution_failure =
    `(
        rust::rust_type
            `($'near_workspaces_result_ExecutionFailure')
            `($'near_workspaces::result::ExecutionFailure')
            "type near_workspaces_result_ExecutionFailure = class end"
    )

:! execution_outcome =
    `(
        rust::rust_type
            `($'near_workspaces_result_ExecutionOutcome')
            `($'near_workspaces::result::ExecutionOutcome')
            "type near_workspaces_result_ExecutionOutcome = class end"
    )

:= sandbox_worker () : resultm::result' (worker sandbox) near_workspaces_error =
    !\($'"near_workspaces::sandbox().await"')

:= dev_deploy
    (wasm : am'::vec u8)
    (worker : worker sandbox)
    : async::future_pin (resultm::result' contract near_workspaces_error)
    =
    := worker = worker |> rust::emit
    !\\(wasm, $'"Box::pin(!worker.dev_deploy(&$0))"')

:= call (fn_name : string) (contract : contract) : call_transaction =
    !\\((contract, fn_name), $'"$0.call(&*$1)"')

:= logs (result : execution_final_result) : am'::vec (rust::ref sm'::str) =
    !\($'"!result.logs()"')

:= into_result
    (result : execution_final_result)
    : resultm::result' execution_success execution_failure
    =
    !\\(result, $'"$0.into_result()"')

:= receipt_failures (result : execution_final_result) : am'::vec (rust::ref execution_outcome) =
    := result = ~~ result
    !\($'"!result.receipt_failures()"')

:= receipt_outcomes (result : execution_final_result) : am'::vec execution_outcome =
    := result = ~~ result
    := result : rust::ref (am'::slice execution_outcome) = !\($'"!result.receipt_outcomes()"')
    result |> rust::into

:= json (result : execution_final_result) : resultm::result' sm'::std_string near_workspaces_error =
    := result = result |> rust::clone_rust
    !\\(result, $'"$0.json()"')

:= borsh (result : execution_final_result) : resultm::result' sm'::std_string near_workspaces_error =
    := result = result |> rust::clone_rust
    !\\(result, $'"$0.borsh()"')

:= total_gas_burnt (result : execution_final_result) : gas =
    !\\(result, $'"$0.total_gas_burnt"')

:= as_gas (gas : gas) : u64 =
    !\\(gas, $'"$0.as_gas()"')

:= outcomes (result : execution_final_result) : am'::vec (rust::ref execution_outcome) =
    := result = result |> rust::emit
    !\($'"!result.outcomes()"')

:= is_success (outcome : execution_outcome) : bool =
    !\\(outcome, $'"$0.is_success()"')

:= gas_burnt (outcome : execution_outcome) : gas =
    !\\(outcome, $'"$0.gas_burnt"')

:= tokens_burnt (outcome : execution_outcome) : near_token_workspaces =
    !\\(outcome, $'"$0.tokens_burnt"')

:= transact
    (call : call_transaction)
    : async::future_pin (resultm::result' execution_final_result near_workspaces_error)
    =
    !\($'"Box::pin(!call.transact())"')

:= gas
    (gas : gas)
    (call : call_transaction)
    : call_transaction
    =
    !\($'"!call.gas(!gas)"')

:= from_tgas
    (tgas : i32)
    : gas
    =
    backend_switch {
        Rust = \ () => !\($'"near_workspaces::types::Gas::from_tgas(!tgas as u64)"') : gas
    }

:= as_yoctonear (tokens : near_token_workspaces) : rust::u128 =
    !\\(tokens, $'"$0.as_yoctonear()"')

:= print_usd retry (result : execution_final_result) =
    := total_gas_burnt = result |> total_gas_burnt |> as_gas
    := total_gas_burnt_usd = total_gas_burnt |> near::gas_to_usd

    trace Info
        \ () => "near_workspaces.print_usd"
        \ () => { retry total_gas_burnt_usd total_gas_burnt }

    result
    |> outcomes
    |> iter::into_iter
    |> iter::cloned
    |> iter::for_each \ outcome =>
        := is_success = outcome |> is_success

        := gas_burnt = outcome |> gas_burnt |> as_gas
        := gas_burnt_usd = gas_burnt |> near::gas_to_usd

        := tokens_burnt = outcome |> tokens_burnt |> as_yoctonear
        := tokens_burnt_usd = tokens_burnt |> near::tokens_to_usd

        trace Info
            \ () => "near_workspaces.print_usd / outcome"
            \ () => { is_success gas_burnt_usd tokens_burnt_usd gas_burnt tokens_burnt }